# TMDB MODELLING

## 🧩 Introduction
In this notebook, we focus on building and evaluating machine learning models to **predict movie revenue** using data collected from the **TMDB API**.  
After completing **data cleaning** and **exploratory data analysis (EDA)** in previous stages, we now aim to develop predictive models that can estimate a movie’s potential box office performance based on features such as **budget**, **popularity**, **runtime**, **genre**, and **language**.

## 🎯 Objective
The main objective of this notebook is to **train, evaluate, and compare multiple regression models** to identify which algorithm best predicts movie revenue.  
The best-performing model will later be integrated into a **machine learning pipeline** for streamlined preprocessing, tuning, and deployment.

## ⚙️ Key Tasks
1. Load the preprocessed dataset (`encoded_movies.csv`).  
2. Split the data into **training** and **testing** sets.  
3. Build and train four regression models:  
   - Linear Regression  
   - Decision Tree Regressor  
   - Random Forest Regressor  
   - XGBoost Regressor  
4. Evaluate model performance using:
   - **R² (Coefficient of Determination)**
   - **MAE (Mean Absolute Error)**
   - **RMSE (Root Mean Squared Error)**
5. Compare all models and select the best performer for pipeline integration.

## ✅ Expected Outcome
By the end of this notebook, we will have:
- A clear understanding of which model performs best for revenue prediction.  
- Quantitative performance metrics (R², MAE, RMSE) for each model.  
- A chosen model ready for **pipeline development and optimization** in the next stage.


In [4]:
#lets import the libraries
import pandas as pd
import numpy as np

In [2]:
#lets load our dataset
df=pd.read_csv('C:/Users/Win/TMDB project/data/encoded_movies.csv')
df.head()


,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western
0,26000000,92691,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,0,0,0,0,0,1,0,0
1,20000000,643612593,156,261.8345,7.793,474,0.30,11,1,0,...,1,0,0,0,0,0,0,1,0,0
2,17647000,3247000,147,271.6580,2.100,8,1.06,12,1,0,...,0,0,0,0,0,0,0,1,0,0
3,55000000,482377782,135,238.2160,7.000,1080,0.17,4,0,0,...,0,0,1,0,0,0,0,0,0,0
4,8500000,11577352,98,150.7366,5.850,70,0.11,4,0,0,...,0,0,1,0,0,0,0,1,0,0


## FEATURE ENGINEERING

In [3]:
#we create a new column budget_popularrity_ratio
df['budget_popularity_ratio']=df['budget']/df['popularity']
df.head(2)

,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio
0,26000000,92691,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,0,0,0,0,1,0,0,88414.082595
1,20000000,643612593,156,261.8345,7.793,474,0.30,11,1,0,...,0,0,0,0,0,0,1,0,0,76384.128142


In [5]:
#we log transform budget and revenue columns to handle skewness
df['revenue']=np.log1p(df['revenue'])
df['budget']=np.log1p(df['budget'])
df.head()

,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio
0,17.073607,11.437037,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,0,0,0,0,1,0,0,88414.082595
1,16.811243,20.282608,156,261.8345,7.793,474,0.30,11,1,0,...,0,0,0,0,0,0,1,0,0,76384.128142
2,16.686076,14.993242,147,271.6580,2.100,8,1.06,12,1,0,...,0,0,0,0,0,0,1,0,0,64960.354563
3,17.822844,19.994238,135,238.2160,7.000,1080,0.17,4,0,0,...,0,1,0,0,0,0,0,0,0,230882.896195
4,15.955577,16.264561,98,150.7366,5.850,70,0.11,4,0,0,...,0,1,0,0,0,0,1,0,0,56389.755375


In [10]:
#lets now bin runtime
# Define bins and labels
bins = [0, 90, 120, 150, df['runtime'].max()]
labels = ['Short', 'Medium', 'Long', 'Very Long']

# Create a new categorical column
df['runtime_bin'] = pd.cut(df['runtime'], bins=bins, labels=labels, include_lowest=True)
df.head()


,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio,runtime_Medium,runtime_Long,runtime_Very Long,runtime_bin
0,17.073607,11.437037,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,1,0,0,88414.082595,True,False,False,Medium
1,16.811243,20.282608,156,261.8345,7.793,474,0.30,11,1,0,...,0,0,1,0,0,76384.128142,False,False,True,Very Long
2,16.686076,14.993242,147,271.6580,2.100,8,1.06,12,1,0,...,0,0,1,0,0,64960.354563,False,True,False,Long
3,17.822844,19.994238,135,238.2160,7.000,1080,0.17,4,0,0,...,0,0,0,0,0,230882.896195,False,True,False,Long
4,15.955577,16.264561,98,150.7366,5.850,70,0.11,4,0,0,...,0,0,1,0,0,56389.755375,True,False,False,Medium


In [11]:
# lets now ecode the runtime bin , using onehot encoding
# Convert boolean dummies to integers
df = pd.get_dummies(df, columns=['runtime_bin'], prefix='runtime', drop_first=True).astype(int)
df.head(2)


,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Thriller,War,Western,budget_popularity_ratio,runtime_Medium,runtime_Long,runtime_Very Long,runtime_Medium,runtime_Long,runtime_Very Long
0,17,11,110,294,6,35,0,4,1,1,...,1,0,0,88414,1,0,0,1,0,0
1,16,20,156,261,7,474,0,11,1,0,...,1,0,0,76384,0,0,1,0,0,1


In [12]:
# we check whether we have categorical columns and see the correlation
df.corr()

,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Thriller,War,Western,budget_popularity_ratio,runtime_Medium,runtime_Long,runtime_Very Long,runtime_Medium,runtime_Long,runtime_Very Long
budget,1.000000,0.568362,0.270746,0.081285,-0.081326,0.337292,-0.374192,-0.185285,0.310755,0.362853,...,-0.083763,0.005136,-0.035128,0.717386,-0.102417,0.185764,0.102677,-0.102417,0.185764,0.102677
revenue,0.568362,1.000000,0.205044,0.042049,0.099679,0.498152,-0.040222,-0.184172,0.135237,0.279006,...,-0.122144,-0.023416,-0.035254,0.422017,-0.112975,0.164511,0.078331,-0.112975,0.164511,0.078331
runtime,0.270746,0.205044,1.000000,0.106363,0.297767,0.282979,-0.038609,0.131000,0.185752,0.065805,...,0.010936,0.158880,0.072289,0.191257,-0.454179,0.475269,0.652015,-0.454179,0.475269,0.652015
popularity,0.081285,0.042049,0.106363,1.000000,0.057828,0.067258,-0.168239,0.049832,0.056233,0.057317,...,0.046621,-0.013699,-0.016427,-0.107941,-0.062844,0.042172,0.090693,-0.062844,0.042172,0.090693
vote_average,-0.081326,0.099679,0.297767,0.057828,1.000000,0.384325,0.142062,0.103310,-0.118361,-0.020953,...,-0.087927,0.092548,0.055219,-0.173825,-0.169884,0.177533,0.180051,-0.169884,0.177533,0.180051
vote_count,0.337292,0.498152,0.282979,0.067258,0.384325,1.000000,0.004523,-0.142119,0.125561,0.206415,...,-0.090993,0.003970,-0.002231,0.239771,-0.167727,0.190930,0.155701,-0.167727,0.190930,0.155701
movie_age,-0.374192,-0.040222,-0.038609,-0.168239,0.142062,0.004523,1.000000,-0.095935,-0.119051,-0.040205,...,-0.033940,0.065819,0.073823,-0.228936,-0.052126,-0.044510,0.024241,-0.052126,-0.044510,0.024241
original_language_encoded,-0.185285,-0.184172,0.131000,0.049832,0.103310,-0.142119,-0.095935,1.000000,0.087547,0.011204,...,-0.033820,0.026070,-0.006805,-0.138534,-0.063715,0.027112,0.128820,-0.063715,0.027112,0.128820
Action,0.310755,0.135237,0.185752,0.056233,-0.118361,0.125561,-0.119051,0.087547,1.000000,0.300408,...,0.178651,0.060095,-0.012378,0.311470,-0.078334,0.167720,0.041006,-0.078334,0.167720,0.041006
Adventure,0.362853,0.279006,0.065805,0.057317,-0.020953,0.206415,-0.040205,0.011204,0.300408,1.000000,...,-0.182698,-0.039655,0.027180,0.406314,-0.105954,0.100948,0.015758,-0.105954,0.100948,0.015758


- **NOW HAVING ENGINEERED ALL THE FEATURURES WE NEED WE MOVE TO MODELLING**

## MODELLING

In [14]:
#WE now split our data into training and testing
from sklearn.model_selection import train_test_split

In [17]:
#we separate the features fro the target
X=df.drop('revenue',axis=1)
y=df['revenue']

In [21]:
#we split the dataset into training and testing sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 1. Linear Regression

In [23]:
#imports
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [24]:
#we initialize the model
lin_reg=LinearRegression()

In [25]:
#we train our model
lin_reg.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [27]:
# lets make the predictions 
y_pred=lin_reg.predict(X_test)

In [30]:
#model evaluations
r2=r2_score(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
rmse=np.sqrt(mean_squared_error(y_test,y_pred))

print(f"R^2:{r2:.4f}")
print(f"MAE:{mae:.4f}")
print(f"RMSE:{rmse:.4f}")

R^2:0.4122
MAE:0.9524
RMSE:1.5472


In [31]:
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_original = np.expm1(y_pred)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_original)
mae = mean_absolute_error(y_test_original, y_pred_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

R²: -0.8118
MAE: 98878627.0884
RMSE: 290627437.1638


## 2. Decision Tree Regressor

In [32]:
#importing the model
from sklearn.tree import DecisionTreeRegressor

In [35]:
#we initialize our model
dt_reg=DecisionTreeRegressor(random_state=42)

In [36]:
#we train our model
dt_reg.fit(X_train,y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [37]:
#we predict using our model
y_pred_dt=dt_reg.predict(X_test)

In [42]:
#evaluation
r2=r2_score(y_test,y_pred_dt)
mae=mean_absolute_error(y_test,y_pred_dt)
rmse=np.sqrt(mean_squared_error(y_test,y_pred_dt))
print(f"R^2:{r2:.4f}")
print(f"MAE:{mae:.4f}")
print(f"RMSE:{rmse:.4f}")

R^2:0.2850
MAE:1.0220
RMSE:1.7064


In [44]:
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_dt_original = np.expm1(y_pred_dt)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_dt_original)
mae = mean_absolute_error(y_test_original, y_pred_dt_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_dt_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

R²: 0.4554
MAE: 72251194.1908
RMSE: 159330826.0393


## 3. Random Forest Regressor

In [47]:
#importing the model
from sklearn.ensemble import RandomForestRegressor

In [49]:
#we initialize or model
rf_reg=RandomForestRegressor(random_state=42)

In [50]:
#we train our model
rf_reg.fit(X_train,y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [52]:
#we make predictions
y_pred_rf=rf_reg.predict(X_test)

In [54]:
#evaluations
r2=r2_score(y_test,y_pred_rf)
mae=mean_absolute_error(y_test,y_pred_rf)
rmse=np.sqrt(mean_squared_error(y_test,y_pred_rf))
print(f"(R^2:{r2:.4f}")
print(f"(MAE:{mae:.4f}")
print(f"(RMSE:{rmse:.4f}")

(R^2:0.4821
(MAE:0.8603
(RMSE:1.4524


In [55]:
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_rf_original = np.expm1(y_pred_rf)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_rf_original)
mae = mean_absolute_error(y_test_original, y_pred_rf_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_rf_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

R²: 0.5202
MAE: 68413319.4823
RMSE: 149554140.9456


## 4. XGBoostregressor

In [56]:
#importing the model
from xgboost import XGBRegressor

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
#we initialize our model
xgb_model=XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [ ]:
#we train our model
xgb_model.fit(X_tarin,y_train)

In [ ]:
#we predict
y_pred_xgb=xgb_model.predict(X_test)

In [ ]:
#evaluations
r2=r2_score(y_test,y_pred_xgb)
mae=mean_absolute_error(y_test,y_pred_xgb)
rmse=np.sqrt(mean_squared_error(y_test,y_pred_xgb))
print(f"(R^2:{r2:.4f}")
print(f"(MAE:{mae:.4f}")
print(f"(RMSE:{rmse:.4f}")

In [ ]:
#evaluation
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_xgb_original = np.expm1(y_pred_xgb)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_xgb_original)
mae = mean_absolute_error(y_test_original, y_pred_xgb_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_xgb_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")


In [ ]:
# INSIGHTS